# Patient-Calibrated 3-Stage XGBoost (v3 - Enhanced)

**Overview: The Patient-Calibrated Paradigm**

This notebook evaluates a patient-calibrated XGBoost pipeline for classifying 3 sleep macro-stages (Wake, NREM, REM) using exclusively wrist accelerometer data.

**Improvements over v1:**
- **44 features** (up from 7): spectral, inter-axis correlations, subject-normalized z-scores, lagged epochs, stillness run length, VM percentile rank, power ratios
- **Pre-sleep data clipping**: drops motion data before first PSG label to reduce memory and compute
- **2D threshold optimization**: searches REM × Wake grid (was REM-only)
- **Minimum bout enforcement**: removes clinically implausible stage transitions < 3 min
- **Enhanced diagnostics**: calibration curves, OOF probability distributions, per-subject scatter plots

**Methodological Note**: All metrics use out-of-fold cross-validation (GroupKFold, 5 splits). The final model fit is used only for feature importance.

In [3]:
!pip install xgboost


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Unzip and Load Data

### Subtask: Raw Data Extraction and Ingestion

This module handles the extraction and ingestion of the raw dataset. We systematically load the high-frequency wrist actigraphy (accelerometer x, y, z) and the corresponding Polysomnography (PSG) labels into isolated Pandas DataFrames. Critically, we intentionally exclude all photoplethysmography (heart rate) data files. This enforces our core research constraint: evaluating the predictive power of computational motion analysis isolated from physiological metrics.

In [4]:
import zipfile
import os
import pandas as pd

# --- 1.A Dataset Extraction ---
# Support both Colab and local paths
zip_path = '/content/heartratedata.zip'
extract_path = '/content/heartratedata/'
if not os.path.exists(zip_path):
    zip_path = r'C:\Users\Ben\OneDrive\Documents\GitHub\Sleep-Time-Model\heartratedata.zip'
    extract_path = r'C:\Users\Ben\OneDrive\Documents\GitHub\Sleep-Time-Model\heartratedata'

if not os.path.exists(extract_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
else:
    print("Extraction skipped: folder already exists.")

base_path = os.path.join(extract_path, 'motion-and-heart-rate-from-a-wrist-worn-wearable-and-labeled-sleep-from-polysomnography-1.0.0')
motion_dir = os.path.join(base_path, 'motion')
labels_dir = os.path.join(base_path, 'labels')

motion_list = []
labels_list = []
subject_ids = []

# --- 1.B Dynamic File Discovery ---
if os.path.exists(motion_dir):
    for filename in os.listdir(motion_dir):
        if filename.endswith('_acceleration.txt'):
            subject_id = filename.split('_')[0]
            subject_ids.append(subject_id)

subject_ids = sorted(subject_ids)
print(f"Discovered {len(subject_ids)} individual subjects.")

# --- 1.C Data Ingestion Loop ---
for subject_id in subject_ids:
    motion_file = os.path.join(motion_dir, f"{subject_id}_acceleration.txt")
    if os.path.exists(motion_file):
        try:
            df_m = pd.read_csv(motion_file, sep=' ', header=None, names=['timestamp', 'x', 'y', 'z'])
            df_m['subject_id'] = subject_id
            motion_list.append(df_m)
        except Exception as e:
            print(f"Error reading motion file for {subject_id}: {e}")

    label_file = os.path.join(labels_dir, f"{subject_id}_labeled_sleep.txt")
    if os.path.exists(label_file):
        try:
            df_l = pd.read_csv(label_file, sep=' ', header=None, names=['timestamp', 'label'])
            df_l['subject_id'] = subject_id
            labels_list.append(df_l)
        except Exception as e:
            print(f"Error reading label file for {subject_id}: {e}")

# --- 1.D DataFrame Concatenation ---
if motion_list:
    motion_df = pd.concat(motion_list, ignore_index=True)
    print("Motion Data successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Motion Data Found.")

if labels_list:
    labels_df = pd.concat(labels_list, ignore_index=True)
    print("PSG Labels successfully loaded into memory.")
else:
    raise ValueError("Critical Error: No Labels Data Found.")

# --- 1.E Clip pre-sleep motion data ---
motion_df['timestamp']  = motion_df['timestamp'].astype(float)
motion_df['subject_id'] = motion_df['subject_id'].astype(str)
labels_df['timestamp']  = labels_df['timestamp'].astype(float)
labels_df['subject_id'] = labels_df['subject_id'].astype(str)
label_start = labels_df.groupby('subject_id')['timestamp'].min().rename('label_start')
motion_df = motion_df.join(label_start, on='subject_id')
motion_df = motion_df[motion_df['timestamp'] >= motion_df['label_start'] - 30].copy()
motion_df.drop(columns='label_start', inplace=True)
print(f"Motion rows after pre-sleep clip: {len(motion_df):,}")

Extraction skipped: folder already exists.
Discovered 31 individual subjects.


MemoryError: Unable to allocate 1.54 GiB for an array with shape (4, 51819151) and data type float64

## 2. Data Synchronization and Label Processing

### Subtask: Temporal Alignment and Macro-Class Grouping

High-frequency raw motion data cannot be directly trained against 30-second sleep epoch labels. This section standardizes the dataset by casting timestamps to compatible formats and employing a backward-filling merge_asof function with a 30-second tolerance guard. For this baseline model, we structurally group the clinical N1, N2, and N3 stages into a single "NREM" macro-class to address clinical practicality and mitigate the extreme class imbalances found in higher-resolution staging.

In [ ]:
label_map = {
    0: 'Wake',
    1: 'NREM',
    2: 'NREM',
    3: 'NREM',
    5: 'REM'
}

labels_df = labels_df[labels_df['label'].isin(label_map.keys())].copy()
labels_df['sleep_stage'] = labels_df['label'].map(label_map)

labels_df['timestamp'] = labels_df['timestamp'].astype(float)
motion_df['subject_id'] = motion_df['subject_id'].astype(str)
labels_df['subject_id'] = labels_df['subject_id'].astype(str)

motion_df = motion_df.sort_values(by='timestamp').reset_index(drop=True)
labels_df = labels_df.sort_values(by='timestamp').reset_index(drop=True)

merged_df = pd.merge_asof(motion_df, labels_df, on='timestamp', by='subject_id',
                           direction='backward', tolerance=30)

merged_df = merged_df.dropna(subset=['sleep_stage'])

print("Class Distribution in Synchronized Dataset:")
print(merged_df['sleep_stage'].value_counts())

## 3. Time-Series Feature Engineering

### Subtask: Contextualizing Motion Data

Accelerometers suffer from the "Stillness Paradox" where REM and NREM stages can present identical instantaneous physical profiles (paralysis vs. deep rest). To address this, we engineer macro-temporal context. By calculating the Vector Magnitude (VM) and generating rolling temporal windows (2 and 5 minutes), we allow the model to recognize "sedentary streaks." We deliberately prune high-frequency noise metrics (Zero-Crossing Rates) to optimize computational efficiency.

In [ ]:
import numpy as np

# --- 3.A Per-Sample Feature Precomputation ---
merged_df['epoch_start'] = (merged_df['timestamp'] // 30) * 30
merged_df['vm'] = np.sqrt(merged_df['x']**2 + merged_df['y']**2 + merged_df['z']**2)
merged_df['tilt_angle'] = np.arctan2(merged_df['z'],
                          np.sqrt(merged_df['x']**2 + merged_df['y']**2))

# --- 3.B Base Epoch Aggregation ---
epoch_df = merged_df.groupby(['subject_id', 'epoch_start']).agg(
    mean_vm=('vm', 'mean'),
    std_vm=('vm', 'std'),
    max_vm=('vm', 'max'),
    range_vm=('vm', lambda x: x.max() - x.min()),
    mean_tilt=('tilt_angle', 'mean'),
    std_tilt=('tilt_angle', 'std'),
    sleep_stage=('sleep_stage', 'first')
).reset_index()

# --- 3.C Spectral Features + ZCR + Inter-Axis Correlations + Spectral Ratios ---
print("Computing spectral + inter-axis + ratio features per epoch...")
records = []
for (sid, es), group in merged_df.groupby(['subject_id', 'epoch_start']):
    vm = group['vm'].values
    x_vals, y_vals, z_vals = group['x'].values, group['y'].values, group['z'].values
    n = len(vm)
    row = {'subject_id': sid, 'epoch_start': es}
    if n >= 8:
        fs = n / 30.0
        fft_vals = np.abs(np.fft.rfft(vm))
        freqs = np.fft.rfftfreq(n, d=1.0 / fs)
        fft_norm = fft_vals / (fft_vals.sum() + 1e-10)
        row['spectral_entropy'] = float(-np.sum(fft_norm * np.log(fft_norm + 1e-10)))
        row['dominant_freq'] = float(freqs[np.argmax(fft_vals[1:]) + 1]) if len(fft_vals) > 1 else 0.0
        row['power_low']  = float(fft_vals[(freqs >= 0.3) & (freqs < 1.0)].sum())
        row['power_high'] = float(fft_vals[(freqs >= 1.0) & (freqs < 3.0)].sum())
        row['power_ratio'] = row['power_high'] / (row['power_low'] + 1e-10)
        row['total_power'] = row['power_low'] + row['power_high']
        vm_c = vm - vm.mean()
        row['zcr'] = float(np.sum(np.diff(np.sign(vm_c)) != 0) / n)
    else:
        row.update({'spectral_entropy': 0.0, 'dominant_freq': 0.0,
                    'power_low': 0.0, 'power_high': 0.0, 'zcr': 0.0,
                    'power_ratio': 0.0, 'total_power': 0.0})
    def safe_corr(a, b):
        if len(a) < 2: return 0.0
        sa, sb = a.std(), b.std()
        if sa == 0 or sb == 0: return 0.0
        return float(np.corrcoef(a, b)[0, 1])
    row['xy_corr'] = safe_corr(x_vals, y_vals)
    row['xz_corr'] = safe_corr(x_vals, z_vals)
    row['yz_corr'] = safe_corr(y_vals, z_vals)
    records.append(row)
extra_df = pd.DataFrame(records)
epoch_df = epoch_df.merge(extra_df, on=['subject_id', 'epoch_start'], how='left')
print(f"Extra features computed. Epoch count: {len(epoch_df):,}")

# --- 3.D Multi-Scale Rolling Windows (2m, 5m, 15m, 30m) ---
epoch_df = epoch_df.sort_values(by=['subject_id', 'epoch_start'])
for window, label in [(4, '2m'), (10, '5m'), (30, '15m'), (60, '30m')]:
    epoch_df[f'roll_mean_vm_{label}'] = epoch_df.groupby('subject_id')['mean_vm'].transform(
        lambda x, w=window: x.rolling(window=w, min_periods=1).mean()
    )
    epoch_df[f'roll_std_vm_{label}'] = epoch_df.groupby('subject_id')['std_vm'].transform(
        lambda x, w=window: x.rolling(window=w, min_periods=1).mean()
    )

# --- 3.E Circadian and Temporal Delta Features ---
epoch_df['time_of_night_hours'] = (
    epoch_df.groupby('subject_id')['epoch_start']
    .transform(lambda x: (x - x.min()) / 3600.0)
)
epoch_df['vm_delta']   = epoch_df.groupby('subject_id')['mean_vm'].diff().fillna(0)
epoch_df['vm_delta_2'] = epoch_df.groupby('subject_id')['mean_vm'].diff(2).fillna(0)

# --- 3.F Dynamic Rest-Activity Thresholding ---
baseline = epoch_df.groupby('subject_id')['mean_vm'].transform(lambda x: x.quantile(0.05))
epoch_df['is_movement'] = epoch_df['mean_vm'] > (baseline + 0.05)
epoch_df['last_movement_time'] = (
    epoch_df['epoch_start'].where(epoch_df['is_movement'])
    .groupby(epoch_df['subject_id']).ffill()
)
epoch_df['time_since_last_movement'] = (
    epoch_df['epoch_start'] - epoch_df['last_movement_time'].fillna(epoch_df['epoch_start'])
)
epoch_df.drop(columns=['is_movement', 'last_movement_time'], inplace=True)

# --- 3.G Fill NaN values ---
epoch_df.columns = epoch_df.columns.astype(str)
epoch_df['std_vm'] = epoch_df['std_vm'].fillna(0)
epoch_df['time_since_last_movement'] = epoch_df['time_since_last_movement'].ffill().fillna(0)
for col in [c for c in epoch_df.columns if c.startswith('roll_')]:
    epoch_df[col] = epoch_df[col].fillna(0)
for col in ['xy_corr', 'xz_corr', 'yz_corr',
            'spectral_entropy', 'dominant_freq', 'power_low', 'power_high', 'zcr',
            'mean_tilt', 'std_tilt', 'power_ratio', 'total_power']:
    if col in epoch_df.columns:
        epoch_df[col] = epoch_df[col].fillna(0)

# --- 3.H Subject-Normalized Z-Score Features ---
print("Computing subject-normalized z-score features...")
motion_features_to_norm = ['mean_vm', 'std_vm', 'max_vm', 'range_vm',
                            'spectral_entropy', 'dominant_freq', 'power_low', 'power_high']
for feat in motion_features_to_norm:
    if feat in epoch_df.columns:
        epoch_df[f'{feat}_znorm'] = epoch_df.groupby('subject_id')[feat].transform(
            lambda x: (x - x.mean()) / (x.std() + 1e-8)
        )

# --- 3.I VM Percentile Rank Within Subject ---
epoch_df['vm_percentile_rank'] = epoch_df.groupby('subject_id')['mean_vm'].rank(pct=True)

# --- 3.J Lagged Epoch Features +/-1, +/-2, +/-3 ---
print("Computing lagged epoch features...")
for lag in [-3, -2, -1, 1, 2, 3]:
    col_name = f'mean_vm_lag_{lag}'
    epoch_df[col_name] = epoch_df.groupby('subject_id')['mean_vm'].shift(-lag)
    epoch_df[col_name] = epoch_df.groupby('subject_id')[col_name].transform(
        lambda x: x.ffill().bfill()
    )

# --- 3.K Stillness Run Length ---
print("Computing stillness run length...")
def compute_stillness_run(group):
    thresh = group['mean_vm'].quantile(0.15)
    still = group['mean_vm'] <= thresh
    run_len = []
    count = 0
    for val in still:
        count = count + 1 if val else 0
        run_len.append(count)
    return pd.Series(run_len, index=group.index)

epoch_df['stillness_run_len'] = epoch_df.groupby('subject_id', group_keys=False).apply(
    compute_stillness_run
)

# Memory cleanup
del merged_df
import gc; gc.collect()
print("\nMemory cleanup: merged_df released.")

print(f"\nFinal epoch_df shape: {epoch_df.shape}")
all_features = [c for c in epoch_df.columns if c not in ['subject_id', 'epoch_start', 'sleep_stage']]
print(f"Total features available: {len(all_features)}")
print(all_features)

## 4. XGBoost Pipeline and Cross-Validation

### Subtask: Gradient Boosting Architecture and Out-of-Fold Evaluation

We deploy an XGBoost classifier with class-frequency-based sample weights for class balancing (replacing SMOTE, which creates temporally impossible synthetic epochs from time-series data). Because XGBoost requires numerical target variables, we use a LabelEncoder. We use `cross_val_predict` with `GroupKFold` to generate **out-of-fold (OOF) predictions** that serve as the basis for all downstream evaluation. The final model fit is performed solely for feature importance extraction.

In [ ]:
import xgboost as xgb
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_sample_weight

# --- 4.A Expanded Feature List (44 features) ---
features = [
    'mean_vm', 'std_vm', 'max_vm', 'range_vm',
    'mean_tilt', 'std_tilt',
    'xy_corr', 'xz_corr', 'yz_corr',
    'spectral_entropy', 'dominant_freq', 'power_low', 'power_high', 'zcr',
    'power_ratio', 'total_power',
    'time_of_night_hours', 'vm_delta', 'vm_delta_2', 'time_since_last_movement',
    'roll_mean_vm_2m', 'roll_std_vm_2m', 'roll_mean_vm_5m', 'roll_std_vm_5m',
    'roll_mean_vm_15m', 'roll_std_vm_15m', 'roll_mean_vm_30m', 'roll_std_vm_30m',
    'mean_vm_znorm', 'std_vm_znorm', 'max_vm_znorm', 'range_vm_znorm',
    'spectral_entropy_znorm', 'dominant_freq_znorm', 'power_low_znorm', 'power_high_znorm',
    'vm_percentile_rank',
    'mean_vm_lag_-3', 'mean_vm_lag_-2', 'mean_vm_lag_-1',
    'mean_vm_lag_1', 'mean_vm_lag_2', 'mean_vm_lag_3',
    'stillness_run_len',
]
target = 'sleep_stage'

# Filter to only features that exist
missing = [f for f in features if f not in epoch_df.columns]
if missing:
    print(f"Warning: {len(missing)} features not found (dropped): {missing}")
features = [f for f in features if f in epoch_df.columns]

X = epoch_df[features].values
le = LabelEncoder()
y_encoded = le.fit_transform(epoch_df[target])
y = epoch_df[target]
groups = epoch_df['subject_id']

print(f"Feature matrix: {X.shape[0]:,} epochs x {X.shape[1]} features")

# --- 4.B XGBoost with class weights (replaces SMOTE) ---
sample_weights = compute_sample_weight('balanced', y_encoded)

xgb_params = dict(
    n_estimators=300,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=5,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    tree_method='hist',
    n_jobs=-1
)

# --- 4.C Patient-Isolated Cross-Validation (manual loop to pass sample_weight) ---
cv = GroupKFold(n_splits=5)
oof_probs = np.zeros((len(y_encoded), 3))

print("\nRunning GroupKFold cross-validation (5 splits, XGBoost + class weights)...")
for fold, (train_idx, test_idx) in enumerate(cv.split(X, y_encoded, groups)):
    model = xgb.XGBClassifier(**xgb_params)
    model.fit(X[train_idx], y_encoded[train_idx], sample_weight=sample_weights[train_idx])
    oof_probs[test_idx] = model.predict_proba(X[test_idx])
    print(f"  Fold {fold+1}/5 complete")
print("Out-of-fold predictions generated.")

# --- 4.D Final Model Fit (for feature importances ONLY) ---
xgb_model = xgb.XGBClassifier(**xgb_params)
xgb_model.fit(X, y_encoded, sample_weight=sample_weights)
class_names = le.classes_
classes = list(class_names)
print(f"\nClasses: {classes}")
print("Final model fitted for feature importance extraction only.")

# --- 4.E Diagnostics: OOF Probability Quantiles ---
import pandas as pd
rem_idx = classes.index('REM')
wake_idx = classes.index('Wake')
nrem_idx = classes.index('NREM')
print("\nOOF Probability Quantiles by Class:")
for cls_name, idx in [('REM', rem_idx), ('Wake', wake_idx), ('NREM', nrem_idx)]:
    q = pd.Series(oof_probs[:, idx]).quantile([0.50, 0.75, 0.90, 0.95, 0.99])
    print(f"  {cls_name}: p50={q.iloc[0]:.3f} p75={q.iloc[1]:.3f} p90={q.iloc[2]:.3f} p95={q.iloc[3]:.3f} p99={q.iloc[4]:.3f}")

## 5. Helper Functions and REM Threshold Optimization

### Subtask: Mitigating the "Stillness Paradox" with Proper Evaluation

We define reusable helper functions for vectorized threshold application and per-subject rolling mode smoothing. The threshold search operates exclusively on out-of-fold probabilities, preventing any data leakage. Rolling mode (not median) is used because sleep stages are categorical, making arithmetic operations like median mathematically invalid.

In [ ]:
from sklearn.metrics import f1_score
from scipy.stats import mode
import numpy as np
import pandas as pd

def apply_thresholds(probs, classes, thresh_rem, thresh_wake):
    rem_idx  = classes.index('REM')
    wake_idx = classes.index('Wake')
    base_preds = np.array(classes)[np.argmax(probs, axis=1)]
    y_pred = base_preds.copy()
    y_pred[probs[:, wake_idx] >= thresh_wake] = 'Wake'
    y_pred[probs[:, rem_idx]  >= thresh_rem]  = 'REM'
    return y_pred

def smooth_predictions(y_pred_strings, subject_ids_array, stage_to_int, int_to_stage, window=5):
    def rolling_mode(series, w=window):
        return series.rolling(window=w, center=True, min_periods=1).apply(
            lambda x: mode(x, keepdims=False).mode
        )
    y_pred_ints = pd.Series(y_pred_strings).map(stage_to_int).values
    temp_df = pd.DataFrame({'subject_id': subject_ids_array, 'pred': y_pred_ints})
    temp_df['smoothed'] = temp_df.groupby('subject_id')['pred'].transform(
        lambda x: rolling_mode(x)
    ).astype(int)
    return temp_df['smoothed'].map(int_to_stage).values

def enforce_min_bout(y_pred_arr, min_epochs=6):
    result = y_pred_arr.copy()
    if len(result) == 0:
        return result
    unique_vals = list(dict.fromkeys(result))
    val_to_int = {v: i for i, v in enumerate(unique_vals)}
    int_arr = np.array([val_to_int[v] for v in result])
    changes = np.where(np.diff(int_arr, prepend=int_arr[0] - 1) != 0)[0]
    for i, start in enumerate(changes):
        end = changes[i + 1] if i + 1 < len(changes) else len(result)
        if (end - start) < min_epochs and start > 0:
            result[start:end] = result[start - 1]
    return result

stage_to_int = {'NREM': 0, 'REM': 1, 'Wake': 2}
int_to_stage = {0: 'NREM', 1: 'REM', 2: 'Wake'}
subject_ids_array = epoch_df['subject_id'].values

rem_thresholds  = np.arange(0.30, 0.71, 0.05)
wake_thresholds = np.arange(0.20, 0.61, 0.05)

best_thresh_rem  = 0.50
best_thresh_wake = 0.50
best_f1_macro    = 0.0

total = len(rem_thresholds) * len(wake_thresholds)
print(f"2D threshold search: {len(rem_thresholds)} REM x {len(wake_thresholds)} Wake = {total} combinations\n")

for thresh_rem in rem_thresholds:
    for thresh_wake in wake_thresholds:
        y_pred = apply_thresholds(oof_probs, classes, thresh_rem, thresh_wake)
        y_pred_smooth = smooth_predictions(y_pred, subject_ids_array, stage_to_int, int_to_stage)
        current_f1 = f1_score(y, y_pred_smooth, average='macro')
        if current_f1 > best_f1_macro:
            best_f1_macro    = current_f1
            best_thresh_rem  = thresh_rem
            best_thresh_wake = thresh_wake

print(f"Optimal REM threshold:  {best_thresh_rem:.2f}")
print(f"Optimal Wake threshold: {best_thresh_wake:.2f}")
print(f"OOF Macro F1 (smoothed, pre-bout-filter): {best_f1_macro:.4f}")

print("\nApplying minimum bout enforcement (min 6 epochs = 3 min)...")
y_pred_thresh = apply_thresholds(oof_probs, classes, best_thresh_rem, best_thresh_wake)
y_pred_smooth = smooth_predictions(y_pred_thresh, subject_ids_array, stage_to_int, int_to_stage)

y_pred_bout = y_pred_smooth.copy()
for sid in epoch_df['subject_id'].unique():
    mask = (epoch_df['subject_id'] == sid).values
    y_pred_bout[mask] = enforce_min_bout(y_pred_smooth[mask])

f1_before = f1_score(y, y_pred_smooth, average='macro')
f1_after  = f1_score(y, y_pred_bout,   average='macro')
print(f"  Macro F1 before bout filter: {f1_before:.4f}")
print(f"  Macro F1 after  bout filter: {f1_after:.4f}")
y_pred_final = y_pred_bout

## 6. Final Evaluation Metrics and Visualization

### Subtask: Out-of-Fold Diagnostic Performance

This section applies the optimized threshold to the out-of-fold probability predictions and generates standard clinical diagnostic metrics. All numbers reported here reflect genuine generalization performance, as no prediction was made on data the model trained on.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (classification_report, confusion_matrix,
                              cohen_kappa_score, roc_auc_score)
from sklearn.calibration import calibration_curve

print(f"3-Stage XGBoost Report (REM Thresh: {best_thresh_rem:.2f}, Wake Thresh: {best_thresh_wake:.2f})")
print("[All metrics from out-of-fold predictions + bout enforcement]\n")
print(classification_report(y, y_pred_final))

kappa = cohen_kappa_score(y, y_pred_final)
print(f"Cohen's Kappa: {kappa:.4f}")

print("\nPer-Class AUC-ROC (One-vs-Rest):")
for i, cls in enumerate(classes):
    y_bin = (y == cls).astype(int)
    auc = roc_auc_score(y_bin, oof_probs[:, i])
    print(f"  AUC ({cls:>4}): {auc:.4f}")

cm = confusion_matrix(y, y_pred_final, labels=classes)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, None]

plt.figure(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=classes, yticklabels=classes)
plt.title('3-Stage XGBoost Confusion Matrix (Out-of-Fold)')
plt.xlabel('Predicted')
plt.ylabel('True')
plt.tight_layout()
plt.show()

importances = xgb_model.feature_importances_
fi_df = pd.DataFrame({'Feature': features, 'Importance': importances}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 10))
sns.barplot(x='Importance', y='Feature', hue='Feature', data=fi_df.head(20),
            palette='viridis', legend=False)
plt.title('Top 20 Feature Importances (XGBoost)')
plt.tight_layout()
plt.show()

print("\nTop 10 features by importance:")
print(fi_df.head(10).to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, cls in enumerate(classes):
    y_bin = (y == cls).astype(int)
    frac_pos, mean_pred = calibration_curve(y_bin, oof_probs[:, i], n_bins=10)
    axes[i].plot(mean_pred, frac_pos, 's-', label='XGBoost')
    axes[i].plot([0, 1], [0, 1], 'k--', label='Perfect')
    axes[i].set_title(f'Calibration: {cls}')
    axes[i].set_xlabel('Mean predicted probability')
    axes[i].set_ylabel('Fraction of positives')
    axes[i].legend()
plt.suptitle('Probability Calibration Curves (Out-of-Fold)')
plt.tight_layout()
plt.show()

report_dict = classification_report(y, y_pred_final, output_dict=True)
macro_f1    = report_dict['macro avg']['f1-score']
accuracy    = report_dict['accuracy']

print(f"\nOut-of-Fold Summary:")
print(f"  Accuracy : {accuracy:.2%}")
print(f"  Macro F1 : {macro_f1:.4f}")
print(f"  Kappa    : {kappa:.4f}")

## 7. Per-Subject Performance Breakdown

To assess variability across individuals, we compute F1 Macro for each subject and report mean and standard deviation. This validates whether "patient calibration" helps uniformly or benefits some subjects more than others.

In [ ]:
from sklearn.metrics import f1_score
import pandas as pd
import matplotlib.pyplot as plt

subject_stats = []
for sid in sorted(epoch_df['subject_id'].unique()):
    mask = (epoch_df['subject_id'] == sid).values
    if mask.sum() == 0:
        continue
    y_true_s = y.values[mask] if hasattr(y, 'values') else y[mask]
    y_pred_s = y_pred_final[mask]
    n_epochs = mask.sum()
    rem_frac  = (y_true_s == 'REM').mean()
    wake_frac = (y_true_s == 'Wake').mean()
    nrem_frac = (y_true_s == 'NREM').mean()
    f1_per_class = f1_score(y_true_s, y_pred_s, average=None,
                             labels=classes, zero_division=0)
    f1_macro = f1_score(y_true_s, y_pred_s, average='macro', zero_division=0)
    subject_stats.append({
        'subject_id': sid, 'n_epochs': n_epochs,
        'rem_frac': rem_frac, 'wake_frac': wake_frac, 'nrem_frac': nrem_frac,
        'f1_macro': f1_macro,
        **{f'f1_{cls}': f1_per_class[i] for i, cls in enumerate(classes)}
    })

subject_df = pd.DataFrame(subject_stats)
print(f"Per-Subject F1 Macro: {subject_df['f1_macro'].mean():.4f} +/- {subject_df['f1_macro'].std():.4f}")
print(f"  Min: {subject_df['f1_macro'].min():.4f}  Max: {subject_df['f1_macro'].max():.4f}")
print()
print(subject_df[['subject_id', 'f1_macro', 'rem_frac', 'wake_frac', 'nrem_frac']].to_string(index=False))

## 8. Hypnogram Visualization

Compare predicted vs. true sleep staging for one example subject. Hypnograms are the standard visualization in sleep medicine and provide intuitive assessment of temporal prediction accuracy.

In [ ]:
import matplotlib.pyplot as plt

# Pick the first subject deterministically
example_subject = sorted(epoch_df['subject_id'].unique())[0]
mask = (epoch_df['subject_id'] == example_subject).values

epochs = epoch_df.loc[mask, 'epoch_start'].values
y_true_subj = y.values[mask] if hasattr(y, 'values') else y[mask]
y_pred_subj = y_pred_final[mask]

stage_order = {'Wake': 2, 'REM': 1, 'NREM': 0}
ytick_labels = ['NREM', 'REM', 'Wake']

true_numeric = [stage_order[s] for s in y_true_subj]
pred_numeric = [stage_order[s] for s in y_pred_subj]

hours = (epochs - epochs[0]) / 3600.0

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
axes[0].step(hours, true_numeric, where='mid', color='blue', linewidth=0.8)
axes[0].set_yticks(range(len(ytick_labels)))
axes[0].set_yticklabels(ytick_labels)
axes[0].set_title(f'True Hypnogram - Subject {example_subject}')
axes[0].set_ylabel('Sleep Stage')

axes[1].step(hours, pred_numeric, where='mid', color='orange', linewidth=0.8)
axes[1].set_yticks(range(len(ytick_labels)))
axes[1].set_yticklabels(ytick_labels)
axes[1].set_title(f'Predicted Hypnogram (OOF) - Subject {example_subject}')
axes[1].set_xlabel('Time (hours)')
axes[1].set_ylabel('Sleep Stage')

plt.tight_layout()
plt.show()